# Temperature Recommendation - Energy-Aware HistGradientBoostingRegressor

Trains the temperature setpoint recommender with the new energy-aware comfort-constrained target.

Artifacts are saved back to Google Drive under `AIModelsAndAlgorithms/TempretureRecomendation`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/VisualizationApp.git'  # <-- change this
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


In [ ]:
# Optional: if your new dataset files are in another Drive folder, set NEW_DATA_DIR and run this cell.
# The files will be linked/copied through DRIVE_DATA_DIR and then used by the cloned repo.
# Example: NEW_DATA_DIR = Path('/content/drive/MyDrive/NewVisualizationAppData')
NEW_DATA_DIR = None

if NEW_DATA_DIR is not None:
    for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
        src = Path(NEW_DATA_DIR) / filename
        dst = DRIVE_DATA_DIR / filename
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            !cp "{src}" "{dst}"
            print('copied', src, '->', dst)
        else:
            print('missing in NEW_DATA_DIR:', src)


In [ ]:
!pip install -q pandas numpy scikit-learn joblib matplotlib


In [ ]:
assert (DATA_DIR / 'temperatureData.csv').exists(), 'Missing temperatureData.csv'
OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'AIModelsAndAlgorithms/TempretureRecomendation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


## Smoke Test
Run this first to verify paths and dependencies without training on the full dataset.


In [ ]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/TempretureRecomendation/hist_gradient_boosting_recommender.py --max-rows 100000 --output-dir "{OUTPUT_DIR}"


## Full Training
Run this cell when you are ready. It may take a long time on the full dataset.


In [ ]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/TempretureRecomendation/hist_gradient_boosting_recommender.py  --output-dir "{OUTPUT_DIR}"


## Check Saved Artifacts


In [ ]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)
